Para esta tarea de text classification usando modelos con transformers, se ha ido consultando como recurso este gran notebook explicativo de Kaggle.

https://www.kaggle.com/code/matleonard/text-classification

Comprobamos versiones y que la CUDA para nuestra gráfica esté bien configurada.

Para mayor referencia, todos los hyperparametros han sido ajustados para esta computadora con estas características:
- GPU nvidia GTX 1060 (6GB)
- CPU intel i7 7th gen
- RAM 16GB DDR4
- SSD 256GB (con poco espacio libre)

Como nuestra VRAM no tiene un tamaño excesivo, el entrenamiento y algunas inferencias se han realizado por batches para evitar errores OOM

In [2]:
import transformers
print(transformers.__version__)

5.14.1


In [3]:
import torch, transformers
print("torch:", torch.__version__)
print("transformers:", transformers.__version__)
print("cuda:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))

torch: 2.5.1+cu121
transformers: 5.14.1
cuda: True
gpu: NVIDIA GeForce GTX 1060


In [ ]:
import os
import re
import numpy as np
import pandas as pd
import torch

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

from datasets import Dataset, DatasetDict
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    DataCollatorWithPadding, TrainingArguments, Trainer
)
import evaluate

# ---------------------------
# 0) Config entorno (Windows/HF)
# ---------------------------
os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"  # opcional
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

print("torch:", torch.__version__)
print("transformers:", __import__("transformers").__version__)
print("cuda:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))

# ---------------------------
# 1) Carga
# ---------------------------
df_over = pd.read_parquet("D:\TFM\data\Overwatch.corpus\overwatch_posts_final.parquet")
df_gam  = pd.read_parquet("D:\TFM\data\gaming.corpus\gaming_posts_final.parquet")
df_cs   = pd.read_parquet("D:\TFM\data\GlobalOffensive.corpus\GlobalOffensive_posts_final.parquet")
df_nsq  = pd.read_parquet("D:\\TFM\\data\\NoStupidQuestions.corpus\\NoStupidQuestions_posts_final.parquet")


# ---------------------------
# 2) Etiquetas
# ---------------------------
df_over["label"] = 1
for d in [df_gam, df_cs, df_nsq]:
    d["label"] = 0

# ---------------------------
# 3) Muestreo balanceado
# ---------------------------
n_pos = min(len(df_over), 80_000)  # puedes subir luego
df_pos = df_over.sample(n=n_pos, random_state=SEED)

df_neg_all = pd.concat([df_gam, df_cs, df_nsq], ignore_index=True)
df_neg = df_neg_all.sample(n=n_pos, random_state=SEED)

df = pd.concat([df_pos, df_neg], ignore_index=True).sample(frac=1, random_state=SEED)

# ---------------------------
# 4) Texto (title + body_text)
# ---------------------------
def clean_text(x):
    x = "" if pd.isna(x) else str(x)
    x = re.sub(r"http\S+|www\.\S+", " ", x)
    x = re.sub(r"\s+", " ", x).strip()
    return x

df["title"] = df["title"].fillna("")
df["body_text"] = df["body_text"].fillna("")
df["text"] = (df["title"] + " [SEP] " + df["body_text"]).map(clean_text)

# quitar vacíos y removed/deleted
df = df[df["text"].str.len() > 5]
df = df[~df["text"].str.lower().isin(["[removed]", "[deleted]"])]

# ---------------------------
# 5) Split
# ---------------------------
train_df, temp_df = train_test_split(df, test_size=0.2, stratify=df["label"], random_state=SEED)
val_df, test_df = train_test_split(temp_df, test_size=0.5, stratify=temp_df["label"], random_state=SEED)

dataset = DatasetDict({
    "train": Dataset.from_pandas(train_df[["text", "label"]], preserve_index=False),
    "validation": Dataset.from_pandas(val_df[["text", "label"]], preserve_index=False),
    "test": Dataset.from_pandas(test_df[["text", "label"]], preserve_index=False),
})

# ---------------------------
# 6) Tokenizer/model
# ---------------------------
model_ckpt = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_ckpt)

MAX_LEN = 160  # más seguro para 6GB

def tokenize(batch):
    return tokenizer(batch["text"], truncation=True, max_length=MAX_LEN)

tokenized = dataset.map(tokenize, batched=True, remove_columns=["text"])
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

model = AutoModelForSequenceClassification.from_pretrained(model_ckpt, num_labels=2)

# ---------------------------
# 7) Métricas
# ---------------------------
acc_metric = evaluate.load("accuracy")
f1_metric = evaluate.load("f1")
prec_metric = evaluate.load("precision")
rec_metric = evaluate.load("recall")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    probs = torch.softmax(torch.tensor(logits), dim=1).numpy()[:, 1]
    out = {
        "accuracy": acc_metric.compute(predictions=preds, references=labels)["accuracy"],
        "f1": f1_metric.compute(predictions=preds, references=labels, average="binary")["f1"],
        "precision": prec_metric.compute(predictions=preds, references=labels, average="binary")["precision"],
        "recall": rec_metric.compute(predictions=preds, references=labels, average="binary")["recall"],
    }
    try:
        out["roc_auc"] = roc_auc_score(labels, probs)
    except:
        out["roc_auc"] = 0.0
    return out

# ---------------------------
# 8) TrainingArguments (Transformers 5.x)
# ---------------------------
args = TrainingArguments(
    output_dir="./ow_topic_cls",
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="steps",
    logging_steps=100,
    learning_rate=2e-5,
    per_device_train_batch_size=4,   # GTX 1060 friendly
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=2,
    num_train_epochs=3,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    fp16=False,                      # más estable en 1060
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["validation"],
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

trainer.train()
print("Test interno:", trainer.evaluate(tokenized["test"]))

# ---------------------------
# 9) Evaluación externa: gold_set_labeled.csv
# ---------------------------
# aqui hubo un problema de ejecucion por que la ruta del gold set estaba mal
# para no reentrenar el modelo, esta parte la ejecuté en otra celda con la ruta corregida
# vamos a dejar esta parte comentada para no tener que hacer inferencia sobre el gold set
# dos veces
'''
gold = pd.read_csv("gold_set_labeled.csv")

# intenta detectar nombres típicos
label_col = "label" if "label" in gold.columns else gold.columns[-1]
if "text" in gold.columns:
    gold["text"] = gold["text"].astype(str)
elif {"title", "body_text"}.issubset(gold.columns):
    gold["text"] = (gold["title"].fillna("") + " [SEP] " + gold["body_text"].fillna("")).astype(str)
else:
    raise ValueError("No encuentro columnas de texto en gold_set_labeled.csv")

gold["text"] = gold["text"].map(clean_text)
gold = gold[gold["text"].str.len() > 5]

# usar solo 0/1 para evaluación principal
gold_bin = gold[gold[label_col].isin([0, 1])].copy()
gold_bin = gold_bin.rename(columns={label_col: "label"})

gold_ds = Dataset.from_pandas(gold_bin[["text", "label"]], preserve_index=False)
gold_tok = gold_ds.map(tokenize, batched=True, remove_columns=["text"])

gold_pred = trainer.predict(gold_tok)
y_true = gold_pred.label_ids
y_pred = np.argmax(gold_pred.predictions, axis=1)
y_prob = torch.softmax(torch.tensor(gold_pred.predictions), dim=1).numpy()[:, 1]

print("\n=== GOLD EXTERNO (0/1) ===")
print(classification_report(y_true, y_pred, digits=4))
print("Confusion matrix:\n", confusion_matrix(y_true, y_pred))
print("ROC-AUC:", roc_auc_score(y_true, y_prob))

# guardar modelo final
trainer.save_model("./ow_topic_cls/best_model")
tokenizer.save_pretrained("./ow_topic_cls/best_model")
print("Modelo guardado en ./ow_topic_cls/best_model")
'''

torch: 2.5.1+cu121
transformers: 5.14.1
cuda: True
gpu: NVIDIA GeForce GTX 1060


Map:   0%|          | 0/127987 [00:00<?, ? examples/s]

Map:   0%|          | 0/15998 [00:00<?, ? examples/s]

Map:   0%|          | 0/15999 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Training Loss,Validation Loss,Epoch,Accuracy,F1,Precision,Recall,Roc Auc
0.272137,0.320577,3,0.920745,0.921389,0.914022,0.928875,0.978628


Test interno: {'eval_loss': 0.32057708501815796, 'eval_accuracy': 0.9207450465654103, 'eval_f1': 0.9213887166769994, 'eval_precision': 0.9140221402214023, 'eval_recall': 0.928875, 'eval_roc_auc': 0.9786282347793472}


FileNotFoundError: [Errno 2] No such file or directory: 'gold_set_labeled.csv'

Como anotación, nuestra anotación manual del gold_set no ha sido binaria, ya que hay ciertos casos que merecen la pena de observar por separado.
- 0 = no es target
- 1 = target
- 2 = falta de contexto para diferenciar (textos muy cortos)
- 3 = error de filtrado (textos con etiquetas placeholder mal filtradas como [deleted])

In [ ]:
#corregimos la ruta y ejecutamos
gold = pd.read_csv("D:\\TFM\\data\\gold_set_labeled.csv")

# columnas reales de tu archivo
text_col = "full_text"
label_col = "manual_label"

# limpieza base
gold[text_col] = gold[text_col].fillna("").astype(str).map(clean_text)

# quitar vacíos y basura obvia
gold = gold[gold[text_col].str.len() > 5]
gold = gold[~gold[text_col].str.strip().str.lower().isin(["[removed]", "[deleted]"])]

# usar solo etiquetas binarias válidas para evaluación principal
gold_bin = gold[gold[label_col].isin([0, 1])].copy()
gold_bin = gold_bin.rename(columns={text_col: "text", label_col: "label"})

print("Gold total:", len(gold))
print("Gold binario (0/1):", len(gold_bin))
print("Distribución labels:\n", gold_bin["label"].value_counts())

gold_ds = Dataset.from_pandas(gold_bin[["text", "label"]], preserve_index=False)
gold_tok = gold_ds.map(tokenize, batched=True, remove_columns=["text"])

gold_pred = trainer.predict(gold_tok)
y_true = gold_pred.label_ids
y_pred = np.argmax(gold_pred.predictions, axis=1)
y_prob = torch.softmax(torch.tensor(gold_pred.predictions), dim=1).numpy()[:, 1]

print("\n=== GOLD EXTERNO (0/1) ===")
print(classification_report(y_true, y_pred, digits=4))
print("Confusion matrix:\n", confusion_matrix(y_true, y_pred))
print("ROC-AUC:", roc_auc_score(y_true, y_prob))

Gold total: 377
Gold binario (0/1): 328
Distribución labels:
 label
0    241
1     87
Name: count, dtype: int64


Map:   0%|          | 0/328 [00:00<?, ? examples/s]


=== GOLD EXTERNO (0/1) ===
              precision    recall  f1-score   support

           0     0.9913    0.9419    0.9660       241
           1     0.8586    0.9770    0.9140        87

    accuracy                         0.9512       328
   macro avg     0.9249    0.9595    0.9400       328
weighted avg     0.9561    0.9512    0.9522       328

Confusion matrix:
 [[227  14]
 [  2  85]]
ROC-AUC: 0.995564458434683


In [9]:
# guardar modelo final
trainer.save_model("./ow_topic_cls/best_model")
tokenizer.save_pretrained("./ow_topic_cls/best_model")
print("Modelo guardado en ./ow_topic_cls/best_model")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Modelo guardado en ./ow_topic_cls/best_model


Hay una pega importante. El gold_set_labeled que hemos clasificado manualmente, proviene de los archivos .parquet donde hemos sampleado los 300.000 ultimos registros (titulo + descripcion de posts) de nuestros corpus.

Como el modelo también ha sido entrenado con varios splits constituidos por la conjunción de estos 4 archivos .parquet, hemos introducido data leakage en la inferencia sobre nuestro gold_set.

Por eso nuestros f1 score no han cambiado practicamente nada entre el split y la inferencia.

Para solucionar esto, en vez de volver a reentrenar el modelo con datos diferentes, lo que sería bastante costoso, lo que vamos a hacer es crear un gold_set v2 con datos que no estén dentro de estos archivos .parquet, para hacer la inferencia de nuevo, aprovechando que tenemos nuestro modelo finetuneado en la carpeta ow_topic_cls.

Vamos a poner a los humanos a trabajar de nuevo!

Para reconstruir nuestro gold set, vamos a aprovechar que no usamos la totalidad de las 300.000 instancias de nuestros archivos .parquet. De este modo, como nuestras random seed fueron fijadas, y esto es un proceso determinista, podemos extraer exactamente los ids que fueron usados para el entrenamiento del modelo. De esta manera podemos excluir de nuestro gold set esos ids ya usados.

Esto además tiene otra ventaja intrinseca, y es que si extrayeramos un gold set nuevo a partir de datos que se encuentren fuera de ese .parquet, el contexto linguistico de todos los subreddits cambiaría simplemente por la diferencia temporal en la que se escribieron esos posts. Aun que el modelo parece bastante robusto, siempre sería conveniente realizar el entrenamiento con datos actualizados para una inferencia actual.

Antes que nada, como no usamos la suma de las 300.000 instancias de cada archivo parquet para el entrenamiento, vamos a comprobar si realmente se han colado en nuestro gold_set o no, ya que podría ser que por suerte no hayan entrado

In [2]:
import pandas as pd
from sklearn.model_selection import train_test_split

SEED = 42

# ---------------------------
# 1) Reconstruir exactamente el mismo pool de entrenamiento (sin reentrenar nada)
# ---------------------------
df_over = pd.read_parquet("D:\\TFM\\data\\Overwatch.corpus\\overwatch_posts_final.parquet")
df_gam  = pd.read_parquet("D:\\TFM\\data\\gaming.corpus\\gaming_posts_final.parquet")
df_cs   = pd.read_parquet("D:\\TFM\\data\\GlobalOffensive.corpus\\GlobalOffensive_posts_final.parquet")
df_nsq  = pd.read_parquet("D:\\TFM\\data\\NoStupidQuestions.corpus\\NoStupidQuestions_posts_final.parquet")

df_over["label"] = 1
for d in [df_gam, df_cs, df_nsq]:
    d["label"] = 0

n_pos = min(len(df_over), 80_000)
df_pos = df_over.sample(n=n_pos, random_state=SEED)

df_neg_all = pd.concat([df_gam, df_cs, df_nsq], ignore_index=True)
df_neg = df_neg_all.sample(n=n_pos, random_state=SEED)

df = pd.concat([df_pos, df_neg], ignore_index=True).sample(frac=1, random_state=SEED)

train_df, temp_df = train_test_split(df, test_size=0.2, stratify=df["label"], random_state=SEED)
val_df, test_df = train_test_split(temp_df, test_size=0.5, stratify=temp_df["label"], random_state=SEED)

train_ids = set(train_df["id"])
val_ids = set(val_df["id"])
test_ids_internal = set(test_df["id"])
seen_ids = train_ids | val_ids | test_ids_internal

print(f"Train: {len(train_ids):,} | Val: {len(val_ids):,} | Test interno: {len(test_ids_internal):,}")
print(f"Total ids vistos (unión): {len(seen_ids):,}")

# ---------------------------
# 2) Cargar el gold set y comparar
# ---------------------------
gold = pd.read_csv("D:\\TFM\\data\\gold_set_labeled.csv")
print(f"\nTotal gold set: {len(gold)}")
print("Columnas disponibles:", list(gold.columns))

gold_ids = set(gold["id"])

overlap_total = gold_ids & seen_ids
overlap_train = gold_ids & train_ids
overlap_val = gold_ids & val_ids
overlap_test_internal = gold_ids & test_ids_internal

print(f"\nOverlap gold vs TOTAL (train+val+test): {len(overlap_total)} de {len(gold_ids)}")
print(f"  - de los cuales en train: {len(overlap_train)}")
print(f"  - de los cuales en val: {len(overlap_val)}")
print(f"  - de los cuales en test interno: {len(overlap_test_internal)}")

# ---------------------------
# 3) Detalle: qué filas del gold set están contaminadas, y de qué estrato
# ---------------------------
if len(overlap_total) > 0:
    gold_contaminado = gold[gold["id"].isin(overlap_total)]
    print(f"\nDistribución de filas contaminadas por estrato:")
    print(gold_contaminado["stratum"].value_counts())
    print(f"\nDistribución de manual_label en filas contaminadas:")
    print(gold_contaminado["manual_label"].value_counts())
else:
    print("\n✅ Sin overlap — el gold set original es válido, no hace falta gold_set_v2.")

Train: 128,000 | Val: 16,000 | Test interno: 16,000
Total ids vistos (unión): 160,000

Total gold set: 380
Columnas disponibles: ['id', 'full_text', 'stratum', 'manual_label']

Overlap gold vs TOTAL (train+val+test): 126 de 380
  - de los cuales en train: 106
  - de los cuales en val: 9
  - de los cuales en test interno: 11

Distribución de filas contaminadas por estrato:
stratum
overwatch        100
hard_negative     10
soft_negative      6
gaming_medio       4
gaming_alto        3
gaming_score0      2
gaming_bajo        1
Name: count, dtype: int64

Distribución de manual_label en filas contaminadas:
manual_label
1    84
0    24
3    16
2     2
Name: count, dtype: int64


Vaya, parece ser que si que han entrado de pleno.

Continuamos con la recostruccion de un gold set nuevo

In [3]:
import pandas as pd
from sklearn.model_selection import train_test_split

SEED = 42

df_over = pd.read_parquet("D:\\TFM\\data\\Overwatch.corpus\\overwatch_posts_final.parquet")
df_gam  = pd.read_parquet("D:\\TFM\\data\\gaming.corpus\\gaming_posts_final.parquet")
df_cs   = pd.read_parquet("D:\\TFM\\data\\GlobalOffensive.corpus\\GlobalOffensive_posts_final.parquet")
df_nsq  = pd.read_parquet("D:\\TFM\\data\\NoStupidQuestions.corpus\\NoStupidQuestions_posts_final.parquet")

df_over["label"] = 1
for d in [df_gam, df_cs, df_nsq]:
    d["label"] = 0

n_pos = min(len(df_over), 80_000)
df_pos = df_over.sample(n=n_pos, random_state=SEED)

df_neg_all = pd.concat([df_gam, df_cs, df_nsq], ignore_index=True)
df_neg = df_neg_all.sample(n=n_pos, random_state=SEED)

df = pd.concat([df_pos, df_neg], ignore_index=True).sample(frac=1, random_state=SEED)

# reproduce el mismo split que en el entrenamiento original
train_df, temp_df = train_test_split(df, test_size=0.2, stratify=df["label"], random_state=SEED)
val_df, test_df = train_test_split(temp_df, test_size=0.5, stratify=temp_df["label"], random_state=SEED)

seen_ids = set(train_df["id"]) | set(val_df["id"]) | set(test_df["id"])
print(f"Total de ids 'vistos' durante entrenamiento/selección de modelo: {len(seen_ids):,}")

Total de ids 'vistos' durante entrenamiento/selección de modelo: 160,000


In [4]:
df_over_avail = df_over[~df_over["id"].isin(seen_ids)]
df_gam_avail  = df_gam[~df_gam["id"].isin(seen_ids)]
df_cs_avail   = df_cs[~df_cs["id"].isin(seen_ids)]
df_nsq_avail  = df_nsq[~df_nsq["id"].isin(seen_ids)]

print(f"Overwatch disponibles: {len(df_over_avail):,} de {len(df_over):,}")
print(f"Gaming disponibles: {len(df_gam_avail):,} de {len(df_gam):,}")
print(f"GlobalOffensive disponibles: {len(df_cs_avail):,} de {len(df_cs):,}")
print(f"NoStupidQuestions disponibles: {len(df_nsq_avail):,} de {len(df_nsq):,}")

# a partir de aquí, reutiliza tu pipeline de muestreo estratificado original
# (clean_text + score_text_fast + buckets para gaming, random sample para el resto)
# sobre estos *_avail en vez de los dfs completos, y etiqueta a mano igual que antes

Overwatch disponibles: 220,000 de 300,000
Gaming disponibles: 273,256 de 300,000
GlobalOffensive disponibles: 273,358 de 300,000
NoStupidQuestions disponibles: 273,386 de 300,000


Recostruimos el 'gold_set_v2.csv'

In [6]:
import re
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

# =========================================================
# 0) CONFIG
# =========================================================
SEED = 42
rng = np.random.default_rng(SEED)

PATH_OVER = r"D:\TFM\data\Overwatch.corpus\overwatch_posts_final.parquet"
PATH_GAM  = r"D:\TFM\data\gaming.corpus\gaming_posts_final.parquet"
PATH_CS   = r"D:\TFM\data\GlobalOffensive.corpus\GlobalOffensive_posts_final.parquet"
PATH_NSQ  = r"D:\TFM\data\NoStupidQuestions.corpus\NoStupidQuestions_posts_final.parquet"

OUT_CSV   = r"D:\TFM\data\gold_set_v2.csv"

# máximo para no cargar RAM en procesamiento de texto
MAX_PER_CORPUS_FOR_BUILD = 120_000

# tamaño final de gold
N_TOTAL = 350
N_POS = 140
N_NEG = N_TOTAL - N_POS  # 210

# =========================================================
# 1) CARGA
# =========================================================
df_over = pd.read_parquet(PATH_OVER)
df_gam  = pd.read_parquet(PATH_GAM)
df_cs   = pd.read_parquet(PATH_CS)
df_nsq  = pd.read_parquet(PATH_NSQ)

# etiquetas de trabajo (solo para reconstruir seen_ids y estratificar)
df_over["label"] = 1
for d in [df_gam, df_cs, df_nsq]:
    d["label"] = 0

# =========================================================
# 2) RECONSTRUIR seen_ids (igual que entrenamiento original)
# =========================================================
n_pos_train = min(len(df_over), 80_000)  # mismo valor que usaste al entrenar
df_pos = df_over.sample(n=n_pos_train, random_state=SEED)

df_neg_all = pd.concat([df_gam, df_cs, df_nsq], ignore_index=True)
df_neg = df_neg_all.sample(n=n_pos_train, random_state=SEED)

df_train_pool = pd.concat([df_pos, df_neg], ignore_index=True).sample(frac=1, random_state=SEED)

train_df, temp_df = train_test_split(
    df_train_pool, test_size=0.2, stratify=df_train_pool["label"], random_state=SEED
)
val_df, test_df = train_test_split(
    temp_df, test_size=0.5, stratify=temp_df["label"], random_state=SEED
)

seen_ids = set(train_df["id"].astype(str)) | set(val_df["id"].astype(str)) | set(test_df["id"].astype(str))
print(f"IDs vistos (train/val/test): {len(seen_ids):,}")

# liberar memoria intermedia
del df_train_pool, df_pos, df_neg, df_neg_all, train_df, val_df, test_df, temp_df

# =========================================================
# 3) EXCLUIR IDs VISTOS (ANTI-LEAKAGE)
# =========================================================
df_over_avail = df_over[~df_over["id"].astype(str).isin(seen_ids)].copy()
df_gam_avail  = df_gam[~df_gam["id"].astype(str).isin(seen_ids)].copy()
df_cs_avail   = df_cs[~df_cs["id"].astype(str).isin(seen_ids)].copy()
df_nsq_avail  = df_nsq[~df_nsq["id"].astype(str).isin(seen_ids)].copy()

print("\nDisponibles tras anti-leakage:")
print(f"Overwatch:         {len(df_over_avail):,}")
print(f"gaming:            {len(df_gam_avail):,}")
print(f"GlobalOffensive:   {len(df_cs_avail):,}")
print(f"NoStupidQuestions: {len(df_nsq_avail):,}")

# =========================================================
# 4) BAJAR TAMAÑO PARA PROCESAR TEXTO (LOW MEMORY)
# =========================================================
def downsample(df, nmax):
    if len(df) <= nmax:
        return df
    return df.sample(n=nmax, random_state=SEED)

df_over_work = downsample(df_over_avail, MAX_PER_CORPUS_FOR_BUILD)
df_gam_work  = downsample(df_gam_avail,  MAX_PER_CORPUS_FOR_BUILD)
df_cs_work   = downsample(df_cs_avail,   MAX_PER_CORPUS_FOR_BUILD)
df_nsq_work  = downsample(df_nsq_avail,  MAX_PER_CORPUS_FOR_BUILD)

print("\nTamaño de trabajo (para construir texto):")
print(f"Overwatch:         {len(df_over_work):,}")
print(f"gaming:            {len(df_gam_work):,}")
print(f"GlobalOffensive:   {len(df_cs_work):,}")
print(f"NoStupidQuestions: {len(df_nsq_work):,}")

# =========================================================
# 5) UTILIDADES TEXTO (LOW MEMORY)
# =========================================================
def clean_text_series(s: pd.Series) -> pd.Series:
    s = s.fillna("").astype(str)
    s = s.str.replace(r"\s+", " ", regex=True).str.strip()
    return s

def build_text_lowmem(df: pd.DataFrame, source_name: str) -> pd.DataFrame:
    # usar solo columnas mínimas
    cols = [c for c in ["id", "title", "body_text", "full_text"] if c in df.columns]
    out = df[cols].copy()

    title = clean_text_series(out["title"]) if "title" in out.columns else pd.Series("", index=out.index)
    body  = clean_text_series(out["body_text"]) if "body_text" in out.columns else pd.Series("", index=out.index)
    full  = clean_text_series(out["full_text"]) if "full_text" in out.columns else pd.Series("", index=out.index)

    tb = (title + " [SEP] " + body).str.strip()
    out["text_for_labeling"] = np.where(tb.str.len() > 5, tb, full)

    out["source_corpus"] = source_name
    # devolver solo lo imprescindible
    return out[["id", "source_corpus", "text_for_labeling"]]

def filter_valid(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    t = out["text_for_labeling"].fillna("")
    out = out[t.str.len() >= 20]  # caracteres, no palabras
    out = out[~t.str.lower().isin(["[removed]", "[deleted]"])]
    return out

# =========================================================
# 6) CONSTRUIR POOLS
# =========================================================
ov = filter_valid(build_text_lowmem(df_over_work, "Overwatch"))
ga = filter_valid(build_text_lowmem(df_gam_work,  "gaming"))
cs = filter_valid(build_text_lowmem(df_cs_work,   "GlobalOffensive"))
ns = filter_valid(build_text_lowmem(df_nsq_work,  "NoStupidQuestions"))

print("\nPools tras limpieza:")
print(f"Overwatch:         {len(ov):,}")
print(f"gaming:            {len(ga):,}")
print(f"GlobalOffensive:   {len(cs):,}")
print(f"NoStupidQuestions: {len(ns):,}")

# =========================================================
# 7) HARD/EASY SAMPLING
# =========================================================
# Keywords para detectar candidatos confusos (hard)
hard_kw = re.compile(
    r"\b(overwatch|ow2|blizzard|hero shooter|support|tank|dps|payload|ranked|competitive|queue|matchmaking|patch|battle\.?net|tracer|genji|mercy)\b",
    flags=re.IGNORECASE
)

def split_hard_easy(df):
    m = df["text_for_labeling"].str.contains(hard_kw, na=False)
    return df[m].copy(), df[~m].copy()

ov_h, ov_e = split_hard_easy(ov)
ga_h, ga_e = split_hard_easy(ga)
cs_h, cs_e = split_hard_easy(cs)
ns_h, ns_e = split_hard_easy(ns)

def sample_n(df, n):
    n = min(int(n), len(df))
    if n <= 0:
        return df.iloc[0:0].copy()
    return df.sample(n=n, random_state=SEED)

def alloc(total, weights_dict):
    keys = list(weights_dict.keys())
    w = np.array([weights_dict[k] for k in keys], dtype=float)
    w = w / w.sum()
    raw = w * total
    base = np.floor(raw).astype(int)
    rem = int(total - base.sum())
    order = np.argsort(-(raw - base))
    for i in order[:rem]:
        base[i] += 1
    return dict(zip(keys, base))

# negativos: 60% hard, 40% easy
N_NEG_H = int(N_NEG * 0.60)
N_NEG_E = N_NEG - N_NEG_H

# reparto por corpus negativo
neg_w = {"gaming": 0.40, "GlobalOffensive": 0.35, "NoStupidQuestions": 0.25}
hard_alloc = alloc(N_NEG_H, neg_w)
easy_alloc = alloc(N_NEG_E, neg_w)

neg_parts = [
    sample_n(ga_h, hard_alloc["gaming"]),
    sample_n(cs_h, hard_alloc["GlobalOffensive"]),
    sample_n(ns_h, hard_alloc["NoStupidQuestions"]),
    sample_n(ga_e, easy_alloc["gaming"]),
    sample_n(cs_e, easy_alloc["GlobalOffensive"]),
    sample_n(ns_e, easy_alloc["NoStupidQuestions"]),
]
neg_sample = pd.concat(neg_parts, ignore_index=True).drop_duplicates(subset=["id"])

# completar si faltan negativos
if len(neg_sample) < N_NEG:
    neg_pool = pd.concat([ga, cs, ns], ignore_index=True)
    neg_pool = neg_pool[~neg_pool["id"].isin(neg_sample["id"])]
    extra = sample_n(neg_pool, N_NEG - len(neg_sample))
    neg_sample = pd.concat([neg_sample, extra], ignore_index=True).drop_duplicates(subset=["id"])

# positivos: 50/50 hard-easy (si no llega, completa del pool)
n_pos_h = int(N_POS * 0.5)
n_pos_e = N_POS - n_pos_h
pos_sample = pd.concat([
    sample_n(ov_h, n_pos_h),
    sample_n(ov_e, n_pos_e)
], ignore_index=True).drop_duplicates(subset=["id"])

if len(pos_sample) < N_POS:
    ov_pool = ov[~ov["id"].isin(pos_sample["id"])]
    extra = sample_n(ov_pool, N_POS - len(pos_sample))
    pos_sample = pd.concat([pos_sample, extra], ignore_index=True).drop_duplicates(subset=["id"])

# combinar final
gold_new = pd.concat([pos_sample, neg_sample], ignore_index=True).drop_duplicates(subset=["id"])

# recortar si pasa de N_TOTAL
if len(gold_new) > N_TOTAL:
    gold_new = gold_new.sample(n=N_TOTAL, random_state=SEED)

# =========================================================
# 8) FORMATO FINAL PARA ETIQUETAR
# =========================================================
gold_new = gold_new.sample(frac=1, random_state=SEED).reset_index(drop=True)

out = pd.DataFrame({
    "id": gold_new["id"].astype(str),
    "full_text": gold_new["text_for_labeling"],          # nombre cómodo para tu pipeline posterior
    "source_corpus": gold_new["source_corpus"],
    "stratum": np.where(gold_new["text_for_labeling"].str.contains(hard_kw, na=False),
                        "hard_candidate", "easy_candidate"),
    "manual_label": "",   # 0=no target, 1=target, 2=falta contexto, 3=mal formato/no usable
    "manual_notes": ""
})

# chequeo anti-leakage
leak = out["id"].isin(seen_ids).sum()
print(f"\nLeakage check (debe ser 0): {leak}")

print("\nDistribución source_corpus:")
print(out["source_corpus"].value_counts())

print("\nDistribución stratum:")
print(out["stratum"].value_counts())

out.to_csv(OUT_CSV, index=False, encoding="utf-8-sig")
print(f"\nGuardado: {OUT_CSV}")
print(f"Filas finales: {len(out)}")

IDs vistos (train/val/test): 160,000

Disponibles tras anti-leakage:
Overwatch:         220,000
gaming:            273,256
GlobalOffensive:   273,358
NoStupidQuestions: 273,386

Tamaño de trabajo (para construir texto):
Overwatch:         120,000
gaming:            120,000
GlobalOffensive:   120,000
NoStupidQuestions: 120,000


C:\Users\LIGHT\AppData\Local\Temp\ipykernel_11008\4006581509.py:125: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  out = out[~t.str.lower().isin(["[removed]", "[deleted]"])]
C:\Users\LIGHT\AppData\Local\Temp\ipykernel_11008\4006581509.py:125: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  out = out[~t.str.lower().isin(["[removed]", "[deleted]"])]
C:\Users\LIGHT\AppData\Local\Temp\ipykernel_11008\4006581509.py:125: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  out = out[~t.str.lower().isin(["[removed]", "[deleted]"])]
C:\Users\LIGHT\AppData\Local\Temp\ipykernel_11008\4006581509.py:125: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  out = out[~t.str.lower().isin(["[removed]", "[deleted]"])]
C:\Users\LIGHT\AppData\Local\Temp\ipykernel_11008\4006581509.py:152: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the


Pools tras limpieza:
Overwatch:         117,426
gaming:            115,881
GlobalOffensive:   115,522
NoStupidQuestions: 119,956


C:\Users\LIGHT\AppData\Local\Temp\ipykernel_11008\4006581509.py:233: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  "stratum": np.where(gold_new["text_for_labeling"].str.contains(hard_kw, na=False),



Leakage check (debe ser 0): 0

Distribución source_corpus:
source_corpus
Overwatch            140
gaming                84
GlobalOffensive       73
NoStupidQuestions     53
Name: count, dtype: int64

Distribución stratum:
stratum
hard_candidate    196
easy_candidate    154
Name: count, dtype: int64

Guardado: D:\TFM\data\gold_set_v2.csv
Filas finales: 350


Una vez hemos clasificado nuestro gold_set_v2_labeled.csv, ya podemos hacer inferencia con nuestro modelo para ver que tal funciona.

In [7]:
import pandas as pd
import numpy as np
import torch
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, roc_auc_score, confusion_matrix

# ---------- Config ----------
MODEL_DIR = r"./ow_topic_cls/best_model"
GOLD_PATH = r"D:\TFM\data\gold_set_v2_labeled.csv"
OUT_PRED  = r"D:\TFM\data\gold_set_v2_with_predictions.csv"
OUT_SUMM  = r"D:\TFM\data\gold_set_v2_metrics_summary.csv"
TEXT_COL = "full_text"
LABEL_COL = "manual_label"
MAX_LEN = 160

# ---------- Cargar modelo ----------
tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_DIR)

# ---------- Cargar gold ----------
df = pd.read_csv(GOLD_PATH)
df[TEXT_COL] = df[TEXT_COL].fillna("").astype(str)
df[LABEL_COL] = pd.to_numeric(df[LABEL_COL], errors="coerce")
df = df[df[TEXT_COL].str.len() > 0].copy()

# ---------- Tokenizar + inferir ----------
def tok(batch):
    return tokenizer(batch["text"], truncation=True, max_length=MAX_LEN)

inf = df.rename(columns={TEXT_COL: "text"}).copy()
ds = Dataset.from_pandas(inf[["text"]], preserve_index=False)
ds_tok = ds.map(tok, batched=True, remove_columns=["text"])

trainer = Trainer(model=model, processing_class=tokenizer)  # transformers 5.x
pred = trainer.predict(ds_tok)

logits = pred.predictions
probs = torch.softmax(torch.tensor(logits), dim=1).numpy()
inf["pred_label"] = np.argmax(logits, axis=1)
inf["pred_proba_1"] = probs[:, 1]

# guardar predicciones completas
out_cols = list(df.columns) + ["pred_label", "pred_proba_1"]
out = df.copy()
out["pred_label"] = inf["pred_label"].values
out["pred_proba_1"] = inf["pred_proba_1"].values
out.to_csv(OUT_PRED, index=False, encoding="utf-8-sig")
print("Predicciones:", OUT_PRED)

# ---------- Métricas 0/1 ----------
m01 = out[out[LABEL_COL].isin([0, 1])].copy()
y_true = m01[LABEL_COL].astype(int).values
y_pred = m01["pred_label"].astype(int).values
y_prob = m01["pred_proba_1"].astype(float).values

acc = accuracy_score(y_true, y_pred)
prec, rec, f1, _ = precision_recall_fscore_support(y_true, y_pred, average="binary", zero_division=0)
auc = roc_auc_score(y_true, y_prob) if len(np.unique(y_true)) > 1 else np.nan
cm = confusion_matrix(y_true, y_pred)

print("\n=== PRINCIPAL (labels 0/1) ===")
print(f"N={len(m01)} | Acc={acc:.4f} | Prec={prec:.4f} | Rec={rec:.4f} | F1={f1:.4f} | AUC={auc:.4f}")
print("Confusion matrix:\n", cm)

# ---------- Análisis 2 y 3 ----------
m2 = out[out[LABEL_COL] == 2]
m3 = out[out[LABEL_COL] == 3]

def aux_stats(x):
    if len(x) == 0:
        return (0, np.nan, np.nan)
    return (len(x), (x["pred_label"] == 1).mean(), x["pred_proba_1"].mean())

n2, ratio1_2, p1_2 = aux_stats(m2)
n3, ratio1_3, p1_3 = aux_stats(m3)

print("\n=== AUXILIAR ===")
print(f"Label=2 (ambiguo): N={n2}, pred1_ratio={ratio1_2:.4f}, mean_p1={p1_2:.4f}" if n2 else "Label=2: N=0")
print(f"Label=3 (ruido):   N={n3}, pred1_ratio={ratio1_3:.4f}, mean_p1={p1_3:.4f}" if n3 else "Label=3: N=0")

# ---------- Resumen CSV ----------
summary = pd.DataFrame([{
    "n_eval_01": len(m01),
    "accuracy_01": acc,
    "precision_01": prec,
    "recall_01": rec,
    "f1_01": f1,
    "roc_auc_01": auc,
    "cm_tn": int(cm[0,0]),
    "cm_fp": int(cm[0,1]),
    "cm_fn": int(cm[1,0]),
    "cm_tp": int(cm[1,1]),
    "n_label2": n2,
    "label2_pred1_ratio": ratio1_2,
    "label2_mean_p1": p1_2,
    "n_label3": n3,
    "label3_pred1_ratio": ratio1_3,
    "label3_mean_p1": p1_3
}])

summary.to_csv(OUT_SUMM, index=False, encoding="utf-8-sig")
print("Resumen métricas:", OUT_SUMM)

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

Map:   0%|          | 0/350 [00:00<?, ? examples/s]

Predicciones: D:\TFM\data\gold_set_v2_with_predictions.csv

=== PRINCIPAL (labels 0/1) ===
N=331 | Acc=0.9063 | Prec=0.8442 | Rec=0.9489 | F1=0.8935 | AUC=0.9682
Confusion matrix:
 [[170  24]
 [  7 130]]

=== AUXILIAR ===
Label=2 (ambiguo): N=7, pred1_ratio=0.8571, mean_p1=0.8433
Label=3 (ruido):   N=12, pred1_ratio=0.8333, mean_p1=0.7998
Resumen métricas: D:\TFM\data\gold_set_v2_metrics_summary.csv


Okay, parece que tanto el F1 como el AUC score han bajado un poco sobre este nuevo gold set, pero de todos modos siguen siendo unos buenos valores.

Frente al modelo anterior con analisis chi2 y reglas lógicas, el modelo de transformers a mejorado mucho, especialmente en recall y F1.

- modelo chi2 -> F1 = 0.674, Precision = 0.649, Recall = 0.701, AUC = 0.866
- transformer -> F1 = 0.894, Precision = 0.844, Recall = 0.949, AUC = 0.968

Esto tiene sentido, ya que el modelo anterior no identificaba bien los casos de contexto implícito, donde no existían palabras con un valor chi2 alto.

Cabe decir que esta inferencia sobre nuestro gold set clasificado manualmente tiene una pega conceptual, y es que la herramienta que estamos tratando de desarrollar, tiene como tarea principal el distinguir a nuestro target, dentro de conjuntos de datos extraidos de Reddit, y no de contextos textuales de todo tipo. 

Es posible que este modelo ajustado haga buenas inferencias con datos textuales de otras plataformas y o redes sociales, pero para obtener resultados óptimos en otro caso de uso (otra plataforma), se debería hacer un fine tuning con datos clasificados procedentes de esa plataforma.

Por último, vamos a hacer una inferencia real de un tamaño mayor, sobre textos que el modelo no ha visto en entrenamiento, para ver como de robustas se mantienen nuestras métricas respecto a una inferencia mayor.

In [8]:
import numpy as np
import pandas as pd
import torch
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, accuracy_score, precision_recall_fscore_support

# -------------------------
# 1) Modelo
# -------------------------
MODEL_DIR = r"./ow_topic_cls/best_model"
tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_DIR)
trainer = Trainer(model=model, processing_class=tokenizer)  # transformers 5.x

MAX_LEN = 160

# -------------------------
# 2) Preparar dataset no-visto
# -------------------------
# Etiquetas "ground truth" por origen
df_over_eval = df_over_avail.copy()
df_over_eval["label"] = 1

df_gam_eval = df_gam_avail.copy()
df_gam_eval["label"] = 0

df_cs_eval = df_cs_avail.copy()
df_cs_eval["label"] = 0

df_nsq_eval = df_nsq_avail.copy()
df_nsq_eval["label"] = 0

df_eval = pd.concat([df_over_eval, df_gam_eval, df_cs_eval, df_nsq_eval], ignore_index=True)

def clean_text_series(s):
    s = s.fillna("").astype(str)
    s = s.str.replace(r"\s+", " ", regex=True).str.strip()
    return s

title = clean_text_series(df_eval["title"]) if "title" in df_eval.columns else pd.Series("", index=df_eval.index)
body  = clean_text_series(df_eval["body_text"]) if "body_text" in df_eval.columns else pd.Series("", index=df_eval.index)
full  = clean_text_series(df_eval["full_text"]) if "full_text" in df_eval.columns else pd.Series("", index=df_eval.index)

tb = (title + " [SEP] " + body).str.strip()
df_eval["text"] = np.where(tb.str.len() > 5, tb, full)

df_eval = df_eval[df_eval["text"].str.len() > 5].copy().reset_index(drop=True)

print("Total evaluación no-vista:", len(df_eval))
print("Distribución labels:\n", df_eval["label"].value_counts())

# -------------------------
# 3) Tokenizar + inferencia
# -------------------------
def tokenize(batch):
    return tokenizer(batch["text"], truncation=True, max_length=MAX_LEN)

ds = Dataset.from_pandas(df_eval[["text", "label"]], preserve_index=False)
ds_tok = ds.map(tokenize, batched=True, remove_columns=["text"])

pred = trainer.predict(ds_tok)

logits = pred.predictions
y_true = pred.label_ids
y_pred = np.argmax(logits, axis=1)
y_prob = torch.softmax(torch.tensor(logits), dim=1).numpy()[:, 1]

# -------------------------
# 4) Métricas
# -------------------------
acc = accuracy_score(y_true, y_pred)
prec, rec, f1, _ = precision_recall_fscore_support(y_true, y_pred, average="binary", zero_division=0)
auc = roc_auc_score(y_true, y_prob)

print("\n=== MÉTRICAS EN CORPUS NO VISTO (AVAIL COMBINADO) ===")
print(f"Accuracy : {acc:.4f}")
print(f"Precision: {prec:.4f}")
print(f"Recall   : {rec:.4f}")
print(f"F1       : {f1:.4f}")
print(f"ROC-AUC  : {auc:.4f}")

print("\nClassification report:")
print(classification_report(y_true, y_pred, digits=4))

print("Confusion matrix:")
print(confusion_matrix(y_true, y_pred))

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

Total evaluación no-vista: 1040000
Distribución labels:
 label
0    820000
1    220000
Name: count, dtype: int64


Map:   0%|          | 0/1040000 [00:00<?, ? examples/s]


=== MÉTRICAS EN CORPUS NO VISTO (AVAIL COMBINADO) ===
Accuracy : 0.9186
Precision: 0.7497
Recall   : 0.9235
F1       : 0.8276
ROC-AUC  : 0.9791

Classification report:
              precision    recall  f1-score   support

           0     0.9781    0.9173    0.9467    820000
           1     0.7497    0.9235    0.8276    220000

    accuracy                         0.9186   1040000
   macro avg     0.8639    0.9204    0.8872   1040000
weighted avg     0.9298    0.9186    0.9215   1040000

Confusion matrix:
[[752169  67831]
 [ 16824 203176]]


Aqui podemos observar que sobre todo nuestra precision ha bajado bastante (0.844 -> 0.750) en la inferencia sobre los datos no vistos en entrenamiento.

Esto puede ser debido al problema que ya intentamos solucionar con nuestro gold set, y es que es muy posible que dentro de los distintos subreddits etiquetados como no target (0), si se esté realmente hablando de Overwatch. De hecho esto fué comprobado en el algoritmo lógico basado en chi2.

Cuando el modelo predice 1 para estos posts, el código lo cuenta como falso positivo, haciendo que nuestro accuracy baje, a pesar de que si se está hablando del target. Es posible que sea un error de etiquetado mas que del modelo.

Esta hipotesis explicaría muy bien el patrón que vemos en los demas scores. El recall se mantiene alto (0.949 -> 0.924) sobre un volumen 3000 veces mayor de datos (331 -> 1,040,000). Es exactamente la firma esperable cuando el 'error' no es error del modelo, sino ruido sistemático de la etiqueta en la dirección 'negativo mal etiquetado'

Para verificar esta hipotesis y estimar cuanto ruido tiene realmente esta etiqueta, vamos a tomar una muestra aleatoria de los falsos positivos para revisarlos a mano.

In [9]:
false_positives_mask = (y_pred == 1) & (y_true == 0)
fp_indices = np.where(false_positives_mask)[0]
print(f"Total falsos positivos: {len(fp_indices):,}")

# Una sola llamada random: selecciona 100 índices (posiciones dentro de fp_indices)
rng = np.random.RandomState(42)
selected_positions = rng.choice(len(fp_indices), size=100, replace=False)
selected_indices = fp_indices[selected_positions]  # ids reales dentro de df_eval / y_prob

# Ahora ambas columnas se construyen a partir de LOS MISMOS índices
sample_fp = df_eval.iloc[selected_indices][["text"]].copy()
sample_fp["pred_proba"] = y_prob[selected_indices]
sample_fp["manual_label"] = ""  # para etiquetar a mano: 1 = sí habla de Overwatch, 0 = no

sample_fp.to_csv("D:\\TFM\\data\\fp_sample_to_verify.csv", index=False)
print(f"Guardado: {len(sample_fp)} falsos positivos para revisión manual")

Total falsos positivos: 67,831
Guardado: 100 falsos positivos para revisión manual


Para no forzar a 0 el contenido que no sea evaluable por revisión manual, o que este mal formateado y tenga texto faltante, vamos a usar el etiquetado ya visto en nuestros gold_set

1 = sí habla de Overwatch (aunque el subreddit lo marcara como negativo)

0 = no habla de Overwatch (falso positivo real del modelo)

2 = ambiguo / contexto insuficiente (no puedes decidir con confianza)

3 = no usable ([deleted], [removed], vacío)

De esta manera, podemos analizar la proporcion de falsos positvos que caen dentro del etiquetado de contenido ambiguo o no evaluable.

In [ ]:
verified = pd.read_csv("D:\\TFM\\data\\fp_sample_to_verify.csv")
verified["manual_label"] = pd.to_numeric(verified["manual_label"], errors="coerce")

print("Distribución de etiquetas en la muestra de FP:")
print(verified["manual_label"].value_counts().sort_index())

# Solo 0/1 entran en el cálculo de "error real vs ruido de etiqueta"
evaluable = verified[verified["manual_label"].isin([0, 1])]
n_total_eval = len(evaluable)

n_real_fp = (evaluable["manual_label"] == 0).sum()     # error real del modelo
n_mislabeled = (evaluable["manual_label"] == 1).sum()  # ruido de la etiqueta de subreddit

print(f"\nEvaluables (0/1): {n_total_eval} de {len(verified)}")
print(f"Errores reales del modelo: {n_real_fp} ({100*n_real_fp/n_total_eval:.1f}%)")
print(f"Ruido de etiqueta (en realidad SÍ es Overwatch): {n_mislabeled} ({100*n_mislabeled/n_total_eval:.1f}%)")

# Categorías 2 y 3 -> reportar aparte, no meterlas en el ratio de error
n_ambiguous = (verified["manual_label"] == 2).sum()
n_unusable = (verified["manual_label"] == 3).sum()
print(f"\nAmbiguos (2): {n_ambiguous} ({100*n_ambiguous/len(verified):.1f}% de los FP muestreados)")
print(f"No usables (3): {n_unusable} ({100*n_unusable/len(verified):.1f}% de los FP muestreados)")

# Precisión corregida, calculada SOLO sobre la proporción evaluable
fp_total = len(fp_indices)
tp_original = (y_pred == 1).sum() - fp_total

fp_reales_estimados = fp_total * (n_real_fp / n_total_eval)
precision_corregida = tp_original / (tp_original + fp_reales_estimados)

print(f"\nPrecisión original (con ruido de etiqueta): {prec:.4f}")
print(f"Precisión corregida (estimada, excluyendo ruido de etiqueta): {precision_corregida:.4f}")

resulta que el modelo comete más "falsos positivos" en textos borrados/cortos que en textos claros, eso es un hallazgo relevante en sí mismo (el modelo predice con más incertidumbre precisamente donde hay menos señal), y lo pierdes si no separas las categorías.

In [10]:
verified = pd.read_csv("D:\\TFM\\data\\fp_sample_verified.csv")
verified["manual_label"] = pd.to_numeric(verified["manual_label"], errors="coerce")

print("Distribución de etiquetas en la muestra de FP:")
print(verified["manual_label"].value_counts().sort_index())

# Solo 0/1 entran en el cálculo de "error real vs ruido de etiqueta"
evaluable = verified[verified["manual_label"].isin([0, 1])]
n_total_eval = len(evaluable)

n_real_fp = (evaluable["manual_label"] == 0).sum()     # error real del modelo
n_mislabeled = (evaluable["manual_label"] == 1).sum()  # ruido de la etiqueta de subreddit

print(f"\nEvaluables (0/1): {n_total_eval} de {len(verified)}")
print(f"Errores reales del modelo: {n_real_fp} ({100*n_real_fp/n_total_eval:.1f}%)")
print(f"Ruido de etiqueta (en realidad SÍ es Overwatch): {n_mislabeled} ({100*n_mislabeled/n_total_eval:.1f}%)")

# Categorías 2 y 3 -> reportar aparte, no meterlas en el ratio de error
n_ambiguous = (verified["manual_label"] == 2).sum()
n_unusable = (verified["manual_label"] == 3).sum()
print(f"\nAmbiguos (2): {n_ambiguous} ({100*n_ambiguous/len(verified):.1f}% de los FP muestreados)")
print(f"No usables (3): {n_unusable} ({100*n_unusable/len(verified):.1f}% de los FP muestreados)")

# Precisión corregida, calculada SOLO sobre la proporción evaluable
fp_total = len(fp_indices)
tp_original = (y_pred == 1).sum() - fp_total

fp_reales_estimados = fp_total * (n_real_fp / n_total_eval)
precision_corregida = tp_original / (tp_original + fp_reales_estimados)

print(f"\nPrecisión original (con ruido de etiqueta): {prec:.4f}")
print(f"Precisión corregida (estimada, excluyendo ruido de etiqueta): {precision_corregida:.4f}")

Distribución de etiquetas en la muestra de FP:
manual_label
0    18
1     7
2    36
3    39
Name: count, dtype: int64

Evaluables (0/1): 25 de 100
Errores reales del modelo: 18 (72.0%)
Ruido de etiqueta (en realidad SÍ es Overwatch): 7 (28.0%)

Ambiguos (2): 36 (36.0% de los FP muestreados)
No usables (3): 39 (39.0% de los FP muestreados)

Precisión original (con ruido de etiqueta): 0.7497
Precisión corregida (estimada, excluyendo ruido de etiqueta): 0.8062


Como podemos observar, de todos estos falsos positivos, tan solo un 25 de 100 posts son evaluables manualmente, es decir, les falta contexto o estan entrando con placeholders como [removed].

Esto tiene una solución sencilla, seguir mejorando el preprocesado del dato, para que entren solo datos ricos en información.

Vamos a hacer una comprobación con un filtro previo para ver si sería posible seguir mejorando las métricas de evaluación.

In [11]:
def has_sufficient_signal(text, min_words=3):
    if not isinstance(text, str):
        return False
    text_clean = text.replace("[deleted]", "").replace("[removed]", "").strip()
    return len(text_clean.split()) >= min_words

df_eval["evaluable"] = df_eval["text"].apply(has_sufficient_signal)
print(df_eval["evaluable"].value_counts())

# Recalcula métricas solo sobre lo que tiene señal mínima
mask = df_eval["evaluable"].values
y_true_filtered = y_true[mask]
y_pred_filtered = y_pred[mask]
y_prob_filtered = y_prob[mask]

from sklearn.metrics import precision_recall_fscore_support, roc_auc_score, accuracy_score
prec_f, rec_f, f1_f, _ = precision_recall_fscore_support(y_true_filtered, y_pred_filtered, average="binary", zero_division=0)
print(f"F1 tras filtrar contenido insuficiente: {f1_f:.4f} (precision={prec_f:.4f}, recall={rec_f:.4f})")

evaluable
True     1025390
False      14610
Name: count, dtype: int64
F1 tras filtrar contenido insuficiente: 0.8344 (precision=0.7589, recall=0.9266)


Vemos que las métricas de evaluación con este filtro simple no mejora apenas, por lo que podemos sacar una conclusión relacionada con el problema ya visto, pero con un enfoque distinto.

El problema es que un posts que nosotros clasificamos como que tiene poco contexto, no depende tanto de la cantidad de palabras que contiene, sino mas bien de si contiene alguna palabra significativa contextualmente o no. Para seguir mejorando el modelo sería necesario usar técnicas de filtrado mas complejas, pero para nuestro caso de uso, un f1 score de 0.89 y una precision de 0.844 sobre nuestro gold set filtrado manualmente y sin leakage es un resultado suficiente.

Además, en el etiquetado manual salta a la vista un detalle para nuestra herramienta al completo que no habiamos tenido en cuenta. Realmente lo que estamos buscando son opiniones acerca de un amplio abanico de caracteristicas del juego, pero en el subreddit de overwatch vemos que hay una gran cantidad de posts que no son un 'rant' propiamente dicho, sino simplemente memes graciosos o gente mostrando clips de jugadas suyas. Creo que encontrar una forma de hacer estos filtros no es facil o es imposible, ya que si que queremos extraer informacion acerca de que es lo que la gente encuentra 'gracioso' por ejemplo, esa informacion suele estar contenida en posts con textos mas cortos. Por lo que definir la linea roja no es sencillo.

De todos modos, planteamos como hipótesis de trabajo que el alto recall y la precisión moderada del clasificador puedan actuar como un filtro implícito adicional en las fases posteriores del pipeline: es esperable que los posts falsos positivos, al no compartir necesariamente la semántica específica de Overwatch, tiendan a agruparse de forma dispersa o en clusters propios claramente diferenciables durante el análisis de temas, en lugar de contaminar los clusters de contenido genuino. Esta hipótesis se valida empíricamente en el capítulo de clusterización (sección X), donde se analiza explícitamente si el contenido identificado como falso positivo en la fase de clasificación aparece mezclado con clusters de temática real de Overwatch o se segrega de forma independiente.